[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/recsys-course/04_neighborhood_linear/04_proyecto_knn_vs_ease.ipynb)

# Proyecto 04 · item-kNN vs EASE para la home de CineMatch
| | |
|---|---|
| **Nivel** | 🟡 Intermedio |
| **Duración** | 3–4 h |
| **GPU** | Opcional (T4 acelera EASE); ≈ 0,5–1 unidad |
| **Prerrequisitos** | Lección 04 y evaluador del módulo 02 |

## 🏢 Contexto de negocio
El equipo de CineMatch tiene en producción un **item-kNN** heredado (coseno, sin tunear) para la
fila «Recomendado para ti». Un *paper* dice que **EASE** es mejor. Tu jefa te pide un informe
riguroso: *¿merece la pena cambiar?* — con métricas de precisión, **cobertura**, **intervalos de
confianza** y coste de memoria/latencia.

## 📦 Dataset
MovieLens 1M, split temporal global 80/20 (utilidades del curso), validación temporal dentro de train.

## 📋 Entregables y rúbrica
| # | Entregable | Criterio |
|---|---|---|
| 1 | `item_knn()` con shrinkage, α y poda top-k | Tests unitarios de la celda pasan |
| 2 | `ease()` en forma cerrada | Diagonal exactamente 0; test de la celda pasa |
| 3 | Tuning **en validación** (grid u Optuna) de ambos | Sin tocar el test hasta el final |
| 4 | Tabla final en test: NDCG@10, Recall@10, Coverage | **EASE ≥ 0,230** y **item-kNN tuneado ≥ 0,220** NDCG@10 (referencia medida: popularidad ≈ 0,214 · kNN heredado ≈ 0,186 · kNN tuneado ≈ 0,227 · EASE ≈ 0,237). Coverage reportada para todos |
| 5 | **IC 95 % por bootstrap pareado** de ΔNDCG (EASE − kNN) y (EASE − popularidad) | Conclusión explícita: ¿es significativo? |
| 6 | Análisis de coste: memoria de B vs S y latencia por usuario | Extrapolación a 50 k y 500 k ítems |

In [ ]:
!pip install -q pyarrow optuna

In [ ]:
import os, random, time, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
warnings.filterwarnings("ignore")
seed = 42; random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
device = "cuda" if torch.cuda.is_available() else "cpu"
FAST_DEV_RUN = True

## 🧰 Utilidades del curso

Para que este notebook sea **autocontenido en Colab**, las tres celdas siguientes
contienen versiones **simplificadas y vectorizadas** de las utilidades que construimos en
[01_data](../01_data) (carga de MovieLens + **split temporal global**) y
[02_evaluation](../02_evaluation) (**NDCG@K, Recall@K, coverage**). Las métricas tienen
las mismas definiciones que `cinematch_eval` (NDCG binario con ganancia lineal, Recall
dividido por $|\mathcal{R}_u|$, promedio sobre todos los usuarios con algún relevante),
pero trabajan con índices y matrices para poder puntuar el catálogo entero por lotes.

Convenciones que usaremos en todo el Bloque II:

| Concepto | Convención |
|---|---|
| Dataset | **MovieLens-1M** (como 01–02; el 00 usa 100K para el EDA) |
| Columnas | `user_id`, `item_id`, `rating`, `timestamp` (IDs originales de MovieLens) |
| Índices internos | `uidx`, `iidx` contiguos `0..n-1`, definidos **solo con train** |
| Split | temporal **global**: el 80 % más antiguo de las interacciones → train, el 20 % más reciente → test (sin val aparte: cuando hay que ajustar, se hace un segundo corte temporal *dentro* de train) |
| Train | **todos** los ratings ("visto" = cualquier rating; los modelos explícitos usan su valor) |
| Relevantes en test | `rating ≥ 4` (la binarización de Liang et al. 2018 / Steck 2019) |
| Evaluación | *full ranking* sobre todo el catálogo, excluyendo lo ya visto en train |

> ⚠️ **Diferencias con `prepare_cinematch` (módulo 01)**: allí el train también se filtraba a
> `rating ≥ 4`, había un 10 % de validación y k-core 5 sobre train. Por eso los números del
> Bloque II **no son comparables** con los de 01–02 (otro dataset, otro protocolo): compara
> siempre modelos dentro del mismo protocolo, como aprendiste en el módulo 02.

In [ ]:
# 🧰 Utilidades del curso (1/3) — carga de MovieLens-1M (como cinematch_data.load_movielens('1m')).
import io, os, zipfile, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.sparse as sp

DATA_DIR = Path(os.environ.get("RECSYS_DATA", "data"))
ML1M_URL = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"
# Espejo público en GitHub (mismo contenido, en CSV) por si GroupLens no responde
ML1M_MIRROR = "https://raw.githubusercontent.com/khanhnamle1994/movielens/master"


def load_movielens_1m(data_dir: Path = DATA_DIR):
    """Devuelve (ratings, movies, users) de MovieLens 1M, cacheados en parquet."""
    d = Path(data_dir) / "ml-1m"
    d.mkdir(parents=True, exist_ok=True)
    cache = {n: d / f"{n}.parquet" for n in ("ratings", "movies", "users")}
    if all(p.exists() for p in cache.values()):
        return tuple(pd.read_parquet(p) for p in cache.values())
    try:
        with urllib.request.urlopen(ML1M_URL, timeout=60) as r:
            zipfile.ZipFile(io.BytesIO(r.read())).extractall(d.parent)
        rd = lambda f, cols: pd.read_csv(d / f, sep="::", engine="python", names=cols, encoding="latin-1")
        ratings = rd("ratings.dat", ["user_id", "item_id", "rating", "timestamp"])
        movies = rd("movies.dat", ["item_id", "title", "genres"])
        users = rd("users.dat", ["user_id", "gender", "age", "occupation", "zip"])
    except Exception as e:  # fallback: espejo en GitHub
        print(f"GroupLens no disponible ({e.__class__.__name__}); usando espejo de GitHub…")
        rd = lambda f: pd.read_csv(f"{ML1M_MIRROR}/{f}", sep="\t", index_col=0, encoding="latin-1")
        ratings = rd("ratings.csv").rename(columns={"movie_id": "item_id"})[["user_id", "item_id", "rating", "timestamp"]]
        movies = rd("movies.csv").rename(columns={"movie_id": "item_id"})[["item_id", "title", "genres"]]
        users = rd("users.csv").rename(columns={"zipcode": "zip"})[["user_id", "gender", "age", "occupation", "zip"]]
    movies["genres"] = movies["genres"].str.split("|")
    movies["year"] = movies["title"].str.extract(r"\((\d{4})\)\s*$").astype(float)
    for n, df in zip(cache, (ratings, movies, users)):
        df.to_parquet(cache[n], index=False)
    return ratings, movies, users

In [ ]:
# 🧰 Utilidades del curso (2/3) — split temporal global (versión 80/20 sin val de cinematch_data.temporal_split).
def temporal_split(df: pd.DataFrame, test_frac: float = 0.2, ts_col: str = "timestamp"):
    """Split temporal GLOBAL: todo lo anterior al corte → train; lo posterior → test.
    El test se filtra a usuarios e ítems vistos en train (warm-start)."""
    cutoff = df[ts_col].quantile(1 - test_frac)
    train, test = df[df[ts_col] <= cutoff], df[df[ts_col] > cutoff]
    test = test[test.user_id.isin(train.user_id) & test.item_id.isin(train.item_id)]
    return train.reset_index(drop=True), test.reset_index(drop=True)


class Encoder:
    """Mapea IDs originales ↔ índices contiguos (ajustado SOLO con train)."""
    def __init__(self, train: pd.DataFrame):
        self.users = np.sort(train.user_id.unique())
        self.items = np.sort(train.item_id.unique())
        self.u2i = pd.Series(np.arange(len(self.users)), index=self.users)
        self.i2i = pd.Series(np.arange(len(self.items)), index=self.items)
        self.n_users, self.n_items = len(self.users), len(self.items)

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df[df.user_id.isin(self.u2i.index) & df.item_id.isin(self.i2i.index)].copy()
        df["uidx"] = self.u2i.loc[df.user_id].values
        df["iidx"] = self.i2i.loc[df.item_id].values
        return df

    def csr(self, df: pd.DataFrame, values=None) -> sp.csr_matrix:
        """Matriz usuario×ítem dispersa (binaria por defecto)."""
        v = np.ones(len(df), dtype=np.float32) if values is None else np.asarray(values, dtype=np.float32)
        return sp.csr_matrix((v, (df.uidx.values, df.iidx.values)), shape=(self.n_users, self.n_items))

In [ ]:
# 🧰 Utilidades del curso (3/3) — métricas top-K vectorizadas (mismas definiciones que cinematch_eval).
from typing import Callable, Dict


def test_relevance(test_enc: pd.DataFrame, min_rating: float = 4.0) -> Dict[int, np.ndarray]:
    """uidx → array de iidx relevantes en test (rating ≥ min_rating)."""
    rel = test_enc[test_enc.rating >= min_rating]
    return {u: g.values for u, g in rel.groupby("uidx")["iidx"]}


def topk_from_scores(scores: np.ndarray, seen: sp.csr_matrix, k: int) -> np.ndarray:
    """Top-K por fila excluyendo ítems ya vistos (scores: batch×n_items, seen: batch×n_items)."""
    scores = np.array(scores, dtype=np.float32, copy=True)
    r, c = seen.nonzero()
    scores[r, c] = -np.inf
    part = np.argpartition(-scores, k, axis=1)[:, :k]
    order = np.take_along_axis(scores, part, 1).argsort(axis=1)[:, ::-1]
    return np.take_along_axis(part, order, 1)


def ndcg_recall_at_k(recs: np.ndarray, relevant: list, k: int):
    """NDCG@K y Recall@K (binarios) para cada usuario."""
    discounts = 1.0 / np.log2(np.arange(2, k + 2))
    ndcg, recall = np.zeros(len(recs)), np.zeros(len(recs))
    for n, (rec, rel) in enumerate(zip(recs, relevant)):
        hits = np.isin(rec[:k], rel)
        idcg = discounts[: min(len(rel), k)].sum()
        ndcg[n] = (hits * discounts).sum() / idcg
        recall[n] = hits.sum() / len(rel)
    return ndcg, recall


def evaluate_topk(score_fn: Callable[[np.ndarray], np.ndarray], train_csr: sp.csr_matrix,
                  relevance: Dict[int, np.ndarray], k: int = 10, batch: int = 512) -> Dict[str, float]:
    """Evalúa un modelo dado por score_fn(uidx_batch) -> scores densos (batch×n_items)."""
    users = np.array(sorted(relevance))
    all_recs = []
    for s in range(0, len(users), batch):
        ub = users[s : s + batch]
        all_recs.append(topk_from_scores(score_fn(ub), train_csr[ub], k))
    recs = np.vstack(all_recs)
    ndcg, rec = ndcg_recall_at_k(recs, [relevance[u] for u in users], k)
    return {f"NDCG@{k}": ndcg.mean(), f"Recall@{k}": rec.mean(),
            "Coverage": len(np.unique(recs)) / train_csr.shape[1], "n_users": len(users)}

In [ ]:
ratings, movies, users = load_movielens_1m()
train, test = temporal_split(ratings)
enc = Encoder(train); X = enc.csr(enc.transform(train)); rel = test_relevance(enc.transform(test))
tr_in, val = temporal_split(train, test_frac=0.1)
enc_v = Encoder(tr_in); Xv = enc_v.csr(enc_v.transform(tr_in)); relv = test_relevance(enc_v.transform(val))
pop = np.asarray(X.sum(0)).ravel().astype(np.float32)
print(X.shape, len(rel), "usuarios test ·", len(relv), "usuarios val")

## Paso 1 · item-kNN  ✏️ TODO
Implementa `item_knn(X, k, shrink, alpha)` que devuelva una matriz dispersa `S` (ítems × ítems) con
`S[j, i]` = similitud coseno asimétrica con shrinkage, diagonal 0 y solo los `k` mayores valores de
cada **columna**. 💡 Pistas: `X.T @ X` disperso → `.tocoo()`; `np.argpartition` por columnas.

In [ ]:
from sklearn.preprocessing import normalize


def item_knn(X: sp.csr_matrix, k: int = 100, shrink: float = 0.0, alpha: float = 0.5) -> sp.csr_matrix:
    # TODO
    raise NotImplementedError


def ease(X: sp.csr_matrix, lam: float = 500.0) -> np.ndarray:
    # TODO: B = I − P · diagMat(1/diag(P)),  P = (XᵀX + λI)⁻¹
    raise NotImplementedError

In [ ]:
# Tests unitarios (ejecútalos tras implementar)
def run_tests():
    Xt = sp.csr_matrix(np.array([[1, 1, 0], [1, 1, 1], [0, 1, 1], [1, 0, 0]], dtype=np.float32))
    S = item_knn(Xt, k=2, shrink=0, alpha=0.5).toarray()
    assert np.allclose(np.diag(S), 0), "la diagonal debe ser 0"
    assert np.isclose(S[0, 1], 2 / np.sqrt(3 * 3)), f"coseno(0,1) esperado 0.667, obtenido {S[0, 1]:.3f}"
    assert (S > 0).sum(0).max() <= 2, "máximo k vecinos por columna"
    B = ease(Xt, lam=1.0)
    assert np.allclose(np.diag(B), 0), "diag(B) debe ser 0"
    G = (Xt.T @ Xt).toarray(); Bref = np.linalg.inv(G + np.eye(3)); Bref = -Bref / np.diag(Bref); np.fill_diagonal(Bref, 0)
    assert np.allclose(B, Bref, atol=1e-4), "EASE no coincide con la referencia"
    print("✅ tests OK")

## Paso 2 · Tuning en validación  ✏️ TODO
Usa Optuna (o un grid) para `k ∈ [20, 1000]`, `shrink ∈ [0, 2000]`, `alpha ∈ [0.3, 1.0]` del kNN y
`λ ∈ [50, 50 000]` (log) de EASE, maximizando NDCG@10 en `(Xv, relv)`. Guarda el historial y dibuja
NDCG vs cada hiperparámetro.

In [ ]:
# TODO: tuning

## Paso 3 · Evaluación final + bootstrap  ✏️ TODO
Re-entrena con `X` (todo train) usando los mejores hiperparámetros y evalúa **una vez** en test.
Para el bootstrap necesitas NDCG **por usuario**: usa `topk_from_scores` + `ndcg_recall_at_k`.
Remuestrea usuarios con reemplazo (B = 2 000) y calcula el percentil 2,5–97,5 de la diferencia.

In [ ]:
# TODO: evaluación final, bootstrap pareado y tabla

## Paso 4 · Coste  ✏️ TODO
Mide memoria (`S.data.nbytes + …` vs `B.nbytes`) y latencia media de `score_fn` por usuario. Extrapola
la memoria a 50 k y 500 k ítems (EASE: $|I|^2\cdot 4$ bytes; kNN: $|I|\cdot k\cdot 8$ bytes aprox.).

In [ ]:
# TODO: análisis de coste

---
# ⛔ SPOILER — intenta resolverlo primero
Solución de referencia completa.

In [ ]:
def prune_topk(S, k):
    S = S.toarray() if sp.issparse(S) else np.asarray(S)
    k = min(k, S.shape[0] - 1)
    rows = np.argpartition(-S, k, axis=0)[:k]
    cols = np.broadcast_to(np.arange(S.shape[1]), rows.shape)
    out = sp.csr_matrix((S[rows, cols].ravel(), (rows.ravel(), cols.ravel())), shape=S.shape)
    out.eliminate_zeros()
    return out


def item_knn(X, k=100, shrink=0.0, alpha=0.5):
    Co = (X.T @ X).tocoo(); n = np.asarray(X.sum(0)).ravel()
    j, i, c = Co.row, Co.col, Co.data.astype(np.float64)
    S = sp.csr_matrix((c / (n[i] ** alpha * n[j] ** (1 - alpha) + shrink), (j, i)), shape=Co.shape)
    S.setdiag(0)
    return prune_topk(S, k)


def ease(X, lam=500.0):
    G = torch.tensor((X.T @ X).toarray(), dtype=torch.float32, device=device)
    P = torch.linalg.inv(G + lam * torch.eye(G.shape[0], device=device))
    B = -P / torch.diag(P); B.fill_diagonal_(0.0)
    return B.cpu().numpy()


knn_fn = lambda Xm, S: (lambda ub: (Xm[ub] @ S).toarray())
ease_fn = lambda Xm, B: (lambda ub: np.asarray(Xm[ub] @ B))
run_tests()

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
n_trials = 15 if FAST_DEV_RUN else 60


def obj_knn(trial):
    k = trial.suggest_int("k", 20, 1000, log=True)
    shrink = trial.suggest_float("shrink", 0.0, 2000.0)
    alpha = trial.suggest_float("alpha", 0.3, 1.0)
    return evaluate_topk(knn_fn(Xv, item_knn(Xv, k, shrink, alpha)), Xv, relv)["NDCG@10"]


def obj_ease(trial):
    lam = trial.suggest_float("lam", 50.0, 50000.0, log=True)
    return evaluate_topk(ease_fn(Xv, ease(Xv, lam)), Xv, relv)["NDCG@10"]


st_knn = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=seed))
st_knn.enqueue_trial({"k": 100, "shrink": 0.0, "alpha": 0.5})          # el kNN «heredado» de producción
st_knn.optimize(obj_knn, n_trials=n_trials)
st_ease = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=seed))
st_ease.optimize(obj_ease, n_trials=max(8, n_trials // 2))
print("kNN :", st_knn.best_params, round(st_knn.best_value, 4))
print("EASE:", st_ease.best_params, round(st_ease.best_value, 4))

In [ ]:
dk = st_knn.trials_dataframe(); de = st_ease.trials_dataframe()
fig, axes = plt.subplots(1, 4, figsize=(17, 3.5))
for ax, col in zip(axes[:3], ["params_k", "params_shrink", "params_alpha"]):
    ax.scatter(dk[col], dk.value, c=dk.number, cmap="viridis"); ax.set_xlabel(col.replace("params_", "")); ax.set_ylabel("NDCG@10 val")
axes[0].set_xscale("log"); axes[3].scatter(de.params_lam, de.value, color="#c44e52"); axes[3].set_xscale("log"); axes[3].set_xlabel("λ (EASE)")
plt.suptitle("Tuning en validación (color = nº de trial)"); plt.tight_layout(); plt.show()

In [ ]:
def per_user_ndcg(score_fn, Xm, relevance, k=10, batch=512):
    us = np.array(sorted(relevance)); out = []
    for s in range(0, len(us), batch):
        ub = us[s:s + batch]; out.append(topk_from_scores(score_fn(ub), Xm[ub], k))
    nd, _ = ndcg_recall_at_k(np.vstack(out), [relevance[u] for u in us], k)
    return nd


bp_k, bp_e = st_knn.best_params, st_ease.best_params
S_legacy = item_knn(X, 100, 0.0, 0.5)
S_best = item_knn(X, bp_k["k"], bp_k["shrink"], bp_k["alpha"])
B_best = ease(X, bp_e["lam"])
models = {"Popularidad": lambda ub: np.tile(pop, (len(ub), 1)), "item-kNN heredado": knn_fn(X, S_legacy),
          "item-kNN tuneado": knn_fn(X, S_best), "EASE tuneado": ease_fn(X, B_best)}
final = pd.DataFrame({m: evaluate_topk(f, X, rel) for m, f in models.items()}).T.drop(columns="n_users")
per_user = {m: per_user_ndcg(f, X, rel) for m, f in models.items()}
final.round(4)

In [ ]:
rng = np.random.default_rng(seed)
n = len(next(iter(per_user.values()))); B_boot = 2000
idx = rng.integers(0, n, size=(B_boot, n))
cis = {}
for a, b in [("EASE tuneado", "item-kNN tuneado"), ("EASE tuneado", "Popularidad"), ("item-kNN tuneado", "item-kNN heredado")]:
    d = per_user[a] - per_user[b]
    boots = d[idx].mean(1)
    cis[f"{a} − {b}"] = (d.mean(), *np.percentile(boots, [2.5, 97.5]))
ci_df = pd.DataFrame(cis, index=["ΔNDCG@10", "IC 2,5 %", "IC 97,5 %"]).T
ci_df["¿significativo?"] = (ci_df["IC 2,5 %"] > 0) | (ci_df["IC 97,5 %"] < 0)
fig, ax = plt.subplots(figsize=(8, 2.8))
ax.errorbar(ci_df["ΔNDCG@10"], range(len(ci_df)), xerr=[ci_df["ΔNDCG@10"] - ci_df["IC 2,5 %"], ci_df["IC 97,5 %"] - ci_df["ΔNDCG@10"]], fmt="o", capsize=5)
ax.axvline(0, color="k", ls="--"); ax.set_yticks(range(len(ci_df)), ci_df.index); ax.set_xlabel("ΔNDCG@10 (IC 95 % bootstrap pareado)")
plt.tight_layout(); plt.show()
ci_df.round(4)

In [ ]:
def latency_ms(f, n_users=256):
    ub = np.array(sorted(rel))[:n_users]; t0 = time.time(); f(ub)
    return (time.time() - t0) / n_users * 1000


mem = {"item-kNN tuneado": S_best.data.nbytes + S_best.indices.nbytes + S_best.indptr.nbytes, "EASE tuneado": B_best.nbytes}
cost = pd.DataFrame({"memoria (MB)": {m: v / 1e6 for m, v in mem.items()},
                     "latencia (ms/usuario, batch)": {m: latency_ms(models[m]) for m in mem}})
k_b = bp_k["k"]
for n_items in [50_000, 500_000]:
    cost.loc["item-kNN tuneado", f"memoria @ {n_items // 1000}k ítems (GB)"] = n_items * k_b * 8 / 1e9
    cost.loc["EASE tuneado", f"memoria @ {n_items // 1000}k ítems (GB)"] = n_items ** 2 * 4 / 1e9
cost.round(3)

### 📝 Informe (solución de referencia)

- Ejecución de referencia (ML-1M, `FAST_DEV_RUN=True`): popularidad ≈ 0,214 · kNN heredado ≈ 0,186 ·
  kNN tuneado ≈ 0,227 · EASE ≈ 0,237 NDCG@10. **EASE gana al kNN tuneado (+0,010) y a la popularidad
  (+0,023) con IC 95 % pareado que excluye el 0**. Revisa siempre la tabla de IC: si el intervalo de una
  diferencia contiene 0, **no** puedes afirmar que un modelo es mejor.
- **Tunear el kNN heredado** da la mayor mejora (+0,04, más que pasar de kNN tuneado a EASE): la lección
  clásica de Dacrema et al. Pero mira **cómo** mejora: Optuna elige shrinkage muy alto y α < 0,5, y la
  coverage se desploma de ≈ 0,22 a ≈ 0,05 — el kNN «tuneado» se ha convertido casi en un recomendador de
  popularidad. EASE consigue más NDCG **y** más coverage (≈ 0,07). Optimizar solo NDCG en un split temporal
  empuja hacia lo popular: por eso coverage (o novedad) debe ser un *guardrail* del A/B.
- **Coste**: con 3,7 k ítems EASE ocupa ~50 MB y es trivial; a 500 k ítems necesitaría ~1 TB → inviable
  sin poda/aproximación. El kNN escala linealmente en el nº de ítems.
- **Recomendación**: llevar EASE a un **A/B test** (módulo 15) contra el kNN heredado, con coverage y
  % de estrenos como *guardrails*; el kNN tuneado es un plan B barato, pero vigila su coverage. Si los
  números de tu ejecución difieren (otra semilla, `FAST_DEV_RUN=False`), la conclusión debe salir de **tu** tabla de IC.

## 🚀 Retos extra
1. Añade **RP3β** y un **ensemble** lineal EASE + kNN (pesos en validación). ¿Gana a ambos con IC > 0?
2. Implementa EASE con **poda** de B a top-k por columna: ¿cuánto NDCG pierdes con k = 100?
3. Repite todo con un split **por usuario** (leave-last-20 %) y compara conclusiones: ¿cambia el ganador?
4. Prueba `BM25Recommender` de `implicit` (pesado BM25 de X) y explica por qué ayuda o no.

## 🤔 Reflexión (producción / MLOps)
- ¿Cada cuánto re-entrenarías EASE? ¿Y el kNN? ¿Qué cambia si quieres reaccionar a la última película vista?
- ¿Qué *guardrails* pondrías en el A/B (coverage, % de estrenos, popularidad media)?
- ¿Cómo registrarías en MLflow los hiperparámetros, la matriz (artefacto) y las métricas con IC?